### 1. Imports and Configuration

In [ ]:
import math
import time

import torch
import torch.nn            as nn
import torch.nn.functional as F

torch.manual_seed(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# -----------------------------------------------------------------------
# Shared configuration across all experiments.
# -----------------------------------------------------------------------

DIM      = 768
N_LAYERS = 12   # real model depth -- used for storage calculations
LR       = 1e-3
BATCH    = 512

print(f'Device : {device}')

### 2. Synthetic Data

In [ ]:
# -----------------------------------------------------------------------
# Hierarchical concept vectors: 4 levels, each level inherits parent features
# and adds new ones. Simulates real semantic relationships.
# -----------------------------------------------------------------------

def build_concepts(n_concepts, feature_noise=0.3):
    concepts = torch.zeros(n_concepts, DIM)
    for i in range(n_concepts):
        level = i % 4
        if level == 0:
            active = torch.randperm(DIM)[:8]
            vals   = torch.randn(len(active)).abs() + 1.0
        elif level == 1:
            parent        = (i // 4) * 4
            parent_active = concepts[parent].nonzero().squeeze(-1)
            inherit       = parent_active[torch.randperm(len(parent_active))[:6]]
            new           = torch.randperm(DIM)[:12]
            active        = torch.cat([inherit, new]).unique()
            vals          = torch.randn(len(active)).abs() + 0.8
        elif level == 2:
            parent        = min((i // 4) * 4 + 1, n_concepts - 1)
            parent_active = concepts[parent].nonzero().squeeze(-1)
            k             = min(10, len(parent_active))
            inherit       = parent_active[torch.randperm(len(parent_active))[:k]]
            new           = torch.randperm(DIM)[:16]
            active        = torch.cat([inherit, new]).unique()
            vals          = torch.randn(len(active)).abs() + 0.6
        else:
            parent        = min((i // 4) * 4 + 2, n_concepts - 1)
            parent_active = concepts[parent].nonzero().squeeze(-1)
            k             = min(14, len(parent_active))
            inherit       = parent_active[torch.randperm(len(parent_active))[:k]]
            new           = torch.randperm(DIM)[:20]
            active        = torch.cat([inherit, new]).unique()
            vals          = torch.randn(len(active)).abs() + 0.4
        active             = active.long().clamp(0, DIM - 1)
        concepts[i, active] = vals[:len(active)]
        concepts[i]        += torch.randn(DIM) * feature_noise
    return F.normalize(concepts, dim=-1)


def make_dir_pairs(concepts, n):
    '''
    Build directional concept pairs.
    Label 0: A is parent of B (A more general).
    Label 1: B is parent of A (B more general).
    Label 2: unrelated.
    '''
    n_concepts = len(concepts)
    xs, ys, labels = [], [], []
    for _ in range(n):
        r = torch.randint(0, 3, (1,)).item()
        if r == 0:
            p = torch.randint(0, n_concepts // 4, (1,)).item() * 4
            c = min(p + torch.randint(1, 4, (1,)).item(), n_concepts - 1)
            xs.append(concepts[p]); ys.append(concepts[c]); labels.append(0)
        elif r == 1:
            p = torch.randint(0, n_concepts // 4, (1,)).item() * 4
            c = min(p + torch.randint(1, 4, (1,)).item(), n_concepts - 1)
            xs.append(concepts[c]); ys.append(concepts[p]); labels.append(1)
        else:
            i, j = torch.randint(0, n_concepts, (2,)).tolist()
            xs.append(concepts[i]); ys.append(concepts[j]); labels.append(2)
    return torch.stack(xs), torch.stack(ys), torch.tensor(labels)


def make_lm_pairs(token_embeddings, cluster_ids, freq, n, seq_len=16, out_classes=3):
    '''Build LM-style next-token prediction pairs from a token embedding table.'''
    xs, labels = [], []
    for _ in range(n):
        seq     = torch.multinomial(freq.expand(seq_len + 1, -1), 1).squeeze(-1)
        weights = torch.arange(1, seq_len + 1).float()
        weights = weights / weights.sum()
        ctx     = (token_embeddings[seq[:-1]] * weights.unsqueeze(-1)).sum(0)
        xs.append(ctx)
        labels.append(cluster_ids[seq[-1].item()].item() % out_classes)
    return torch.stack(xs), torch.tensor(labels)


def generate_sequences(vocab, n_clusters, n_seqs, seq_len):
    '''Markov sequences with cluster-structured transition probabilities.'''
    tpc        = vocab // n_clusters
    transition = torch.zeros(vocab, vocab)
    for c in range(n_clusters):
        s, e   = c * tpc, (c + 1) * tpc
        transition[s:e, s:e] = 1.0
        nc     = (c + 1) % n_clusters
        ns, ne = nc * tpc, (nc + 1) * tpc
        transition[s:e, ns:ne] = 0.3
        rc     = (c + 3) % n_clusters
        rs, re = rc * tpc, (rc + 1) * tpc
        transition[s:e, rs:re] = 0.1
    transition = transition / transition.sum(dim=-1, keepdim=True)
    seqs       = torch.zeros(n_seqs, seq_len + 1, dtype=torch.long)
    seqs[:, 0] = torch.randint(0, vocab, (n_seqs,))
    for t in range(seq_len):
        probs          = transition[seqs[:, t]]
        seqs[:, t + 1] = torch.multinomial(probs, 1).squeeze(-1)
    return seqs

### 3. Tversky Projection Models

In [ ]:
# -----------------------------------------------------------------------
# Projection-style models: map a single embedding vector to OUT_DIM scores.
# Used in the early feature-sweep experiment (Section 8).
# -----------------------------------------------------------------------

class LinearProj(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
    def forward(self, x, shared_features=None):
        return x @ self.w.t()
    def param_count(self):
        return self.w.numel()


class TverskyProj(nn.Module):
    '''
    Tversky similarity projection.
    If shared_features is None (default), the module owns its feature bank.
    Otherwise the external tensor is used (for cross-layer sharing).
    '''
    def __init__(self, in_dim, out_dim, num_features, shared_features=None):
        super().__init__()
        self.out_dim       = out_dim
        self.owns_features = shared_features is None
        if self.owns_features:
            self.features  = nn.Parameter(torch.empty(num_features, in_dim).uniform_(-0.02, 0.02))
        else:
            self.register_buffer('features', shared_features)
        self.prototypes = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta      = nn.Parameter(torch.tensor(1.0))
        self.alpha      = nn.Parameter(torch.tensor(0.5))
        self.beta       = nn.Parameter(torch.tensor(0.5))

    def _compute(self, x, feat, proto):
        x_f = x     @ feat.t()
        p_f = proto @ feat.t()
        x_s = torch.sigmoid(x_f * 5.0)
        p_s = torch.sigmoid(p_f * 5.0)
        x_a = x_f * x_s
        p_a = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t * (x_a @ p_a.t()) - a * (x_a @ (1 - p_s).t()) - b * ((1 - x_s) @ p_a.t())

    def forward(self, x, shared_features=None):
        feat = shared_features if shared_features is not None else self.features
        return self._compute(x, feat, self.prototypes)

    def param_count(self):
        base = self.prototypes.numel() + 3
        if self.owns_features:
            base += self.features.numel()
        return base


class TverskyHardThresh(TverskyProj):
    '''TverskyProj with hard threshold membership (non-differentiable).'''
    def forward(self, x, shared_features=None):
        feat  = shared_features if shared_features is not None else self.features
        proto = self.prototypes
        x_f   = x     @ feat.t()
        p_f   = proto @ feat.t()
        x_s   = (x_f > 0).float()
        p_s   = (p_f > 0).float()
        x_a   = x_f * x_s
        p_a   = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t * (x_a @ p_a.t()) - a * (x_a @ (1 - p_s).t()) - b * ((1 - x_s) @ p_a.t())


class TverskyPolyApprox(TverskyProj):
    '''TverskyProj with piecewise-linear sigmoid approximation.'''
    def forward(self, x, shared_features=None):
        feat  = shared_features if shared_features is not None else self.features
        proto = self.prototypes
        x_f   = x     @ feat.t()
        p_f   = proto @ feat.t()
        x_s   = torch.clamp(x_f * 5.0 / 4 + 0.5, 0, 1)
        p_s   = torch.clamp(p_f * 5.0 / 4 + 0.5, 0, 1)
        x_a   = x_f * x_s
        p_a   = p_f * p_s
        t, a, b = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t * (x_a @ p_a.t()) - a * (x_a @ (1 - p_s).t()) - b * ((1 - x_s) @ p_a.t())


class AsymmetricLinearV1(nn.Module):
    '''Asymmetric via presence/absence split. Same params as linear + 1 scalar.'''
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w     = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
        self.alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, shared_features=None):
        out = x @ self.w.t()
        return torch.relu(out) - self.alpha.abs() * torch.relu(-out)
    def param_count(self):
        return self.w.numel() + 1


class AsymmetricLinearV2(nn.Module):
    '''Asymmetric via signed gate. Same params as linear + 1 scalar.'''
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w     = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
        self.alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, shared_features=None):
        out  = x @ self.w.t()
        gate = torch.sign(out)
        return out * (1 + self.alpha.abs() * gate)
    def param_count(self):
        return self.w.numel() + 1

### 4. Directional Classifier Models

In [ ]:
# -----------------------------------------------------------------------
# Classifier models: map a concatenated pair (IN = DIM * 2) to OUT classes.
# Used in directional classification experiments (Sections 9, 10).
# -----------------------------------------------------------------------

class LinearCls(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_dim, in_dim) * 0.02)
    def forward(self, x):
        return x @ self.w.t()
    def param_count(self):
        return self.w.numel()


class _TverskyBase(nn.Module):
    '''Shared forward logic for all Tversky classifier variants.'''
    def _tversky_fwd(self, x, feat, proto):
        x_f = x     @ feat.t()
        p_f = proto @ feat.t()
        x_s = self._membership(x_f)
        p_s = self._membership(p_f)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b  = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t * (x_a @ p_a.t()) - a * (x_a @ (1 - p_s).t()) - b * ((1 - x_s) @ p_a.t())

    def _membership(self, t):
        return torch.sigmoid(t * 5.0)


class TverskySigmoid(_TverskyBase):
    '''Tversky with sigmoid membership and a separate feature bank.'''
    def __init__(self, in_dim, out_dim, nf):
        super().__init__()
        self.nf         = nf
        self.features   = nn.Parameter(torch.empty(nf, in_dim).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta      = nn.Parameter(torch.tensor(1.0))
        self.alpha      = nn.Parameter(torch.tensor(0.5))
        self.beta       = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        return self._tversky_fwd(x, self.features, self.prototypes)
    def param_count(self):
        return self.features.numel() + self.prototypes.numel() + 3


class TverskyReLU(_TverskyBase):
    '''Tversky with relu-normalised membership (no exp, cheaper than sigmoid).'''
    def __init__(self, in_dim, out_dim, nf):
        super().__init__()
        self.nf         = nf
        self.features   = nn.Parameter(torch.empty(nf, in_dim).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta      = nn.Parameter(torch.tensor(1.0))
        self.alpha      = nn.Parameter(torch.tensor(0.5))
        self.beta       = nn.Parameter(torch.tensor(0.5))
    def _membership(self, t):
        pos = torch.relu(t)
        return pos / pos.amax(dim=-1, keepdim=True).clamp(min=1e-8)
    def forward(self, x):
        return self._tversky_fwd(x, self.features, self.prototypes)
    def param_count(self):
        return self.features.numel() + self.prototypes.numel() + 3


class TverskyHardSTE(_TverskyBase):
    '''Tversky with hard threshold membership + straight-through estimator.'''
    def __init__(self, in_dim, out_dim, nf):
        super().__init__()
        self.nf         = nf
        self.features   = nn.Parameter(torch.empty(nf, in_dim).uniform_(-0.02, 0.02))
        self.prototypes = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta      = nn.Parameter(torch.tensor(1.0))
        self.alpha      = nn.Parameter(torch.tensor(0.5))
        self.beta       = nn.Parameter(torch.tensor(0.5))
    def _membership(self, t):
        hard = (t > 0).float()
        return hard + (t.sigmoid() - t.sigmoid().detach())
    def forward(self, x):
        return self._tversky_fwd(x, self.features, self.prototypes)
    def param_count(self):
        return self.features.numel() + self.prototypes.numel() + 3


class TverskyNoFeatures(_TverskyBase):
    '''
    Tversky without a separate feature bank.
    Prototypes serve as their own feature space via cross-similarity.
    Same storage as linear + 3 scalars.
    '''
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.prototypes = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta      = nn.Parameter(torch.tensor(1.0))
        self.alpha      = nn.Parameter(torch.tensor(0.5))
        self.beta       = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        p_norm = F.normalize(self.prototypes, dim=-1)
        return self._tversky_fwd(x, self.prototypes, p_norm)
    def _tversky_fwd(self, x, feat, proto):
        x_f = x     @ feat.t()
        p_f = proto @ proto.t()
        x_s = self._membership(x_f)
        p_s = self._membership(p_f)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b  = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t * (x_a @ p_a.t()) - a * (x_a @ (1 - p_s).t()) - b * ((1 - x_s) @ p_a.t())
    def param_count(self):
        return self.prototypes.numel() + 3


class TverskyNoFeatPoly(TverskyNoFeatures):
    '''TverskyNoFeatures with piecewise-linear sigmoid approximation.'''
    def _membership(self, t):
        return torch.clamp(t * 5.0 / 4 + 0.5, 0, 1)


class TverskyNoFeatLearnedScale(TverskyNoFeatures):
    '''TverskyNoFeatures with learned sigmoid steepness.'''
    def __init__(self, in_dim, out_dim):
        super().__init__(in_dim, out_dim)
        self.scale = nn.Parameter(torch.tensor(5.0))
    def _membership(self, t):
        s = self.scale.abs().clamp(min=0.1)
        return torch.sigmoid(t * s)
    def param_count(self):
        return self.prototypes.numel() + 4


class TverskyLowRank(_TverskyBase):
    '''
    Tversky with low-rank prototype factorization W = U @ V.
    Saves storage and speeds up matmuls at large in_dim.
    '''
    def __init__(self, in_dim, out_dim, rank=32):
        super().__init__()
        self.U     = nn.Parameter(torch.randn(out_dim, rank) * 0.02)
        self.V     = nn.Parameter(torch.randn(rank, in_dim) * 0.02)
        self.theta = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        W      = self.U @ self.V
        p_norm = F.normalize(W, dim=-1)
        return self._tversky_fwd(x, W, p_norm)
    def _tversky_fwd(self, x, feat, proto):
        x_f = x     @ feat.t()
        p_f = proto @ proto.t()
        x_s = self._membership(x_f)
        p_s = self._membership(p_f)
        x_a, p_a = x_f * x_s, p_f * p_s
        t, a, b  = self.theta.abs(), self.alpha.abs(), self.beta.abs()
        return t * (x_a @ p_a.t()) - a * (x_a @ (1 - p_s).t()) - b * ((1 - x_s) @ p_a.t())
    def param_count(self):
        return self.U.numel() + self.V.numel() + 3


class TverskySharedCls(_TverskyBase):
    '''Tversky classifier with externally provided shared feature bank.'''
    def __init__(self, in_dim, out_dim, nf, shared_feat):
        super().__init__()
        self.shared_feat = shared_feat
        self.nf          = nf
        self.prototypes  = nn.Parameter(torch.empty(out_dim, in_dim).uniform_(-0.02, 0.02))
        self.theta       = nn.Parameter(torch.tensor(1.0))
        self.alpha       = nn.Parameter(torch.tensor(0.5))
        self.beta        = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        return self._tversky_fwd(x, self.shared_feat, self.prototypes)
    def param_count(self):
        return self.prototypes.numel() + 3

### 5. Linear Layer Variants

In [ ]:
# -----------------------------------------------------------------------
# Alternative linear layer implementations benchmarked in Section 10.
# All share the interface: forward(x), param_count(), ternary_bytes().
# -----------------------------------------------------------------------

class StandardLinear(nn.Module):
    '''Full dense linear.'''
    def __init__(self, in_f, out_f):
        super().__init__()
        self.w = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
    def forward(self, x):
        return x @ self.w.t()
    def param_count(self):
        return self.w.numel()
    def ternary_bytes(self):
        return self.w.numel() * 1.6 / 8


class LowRankLinear(nn.Module):
    '''W = A @ B, rank r controls compression.'''
    def __init__(self, in_f, out_f, rank=64):
        super().__init__()
        self.A = nn.Parameter(torch.randn(out_f, rank) * 0.02)
        self.B = nn.Parameter(torch.randn(rank, in_f) * 0.02)
    def forward(self, x):
        return (x @ self.B.t()) @ self.A.t()
    def param_count(self):
        return self.A.numel() + self.B.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class BlockDiagonalLinear(nn.Module):
    '''n_blocks independent [block_out, block_in] matrices, batched via einsum.'''
    def __init__(self, in_f, out_f, n_blocks=8):
        super().__init__()
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks  = n_blocks
        self.block_in  = in_f  // n_blocks
        self.block_out = out_f // n_blocks
        self.blocks    = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
    def forward(self, x):
        bsz     = x.shape[:-1]
        x_b     = x.reshape(*bsz, self.n_blocks, self.block_in)
        out     = torch.einsum('...bi,boi->...bo', x_b, self.blocks)
        return out.reshape(*bsz, self.n_blocks * self.block_out)
    def param_count(self):
        return self.blocks.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class BlockDiagPlusMix(nn.Module):
    '''Block-diagonal + low-rank cross-block mixing to recover cross-group interaction.'''
    def __init__(self, in_f, out_f, n_blocks=8, mix_rank=16):
        super().__init__()
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks  = n_blocks
        self.block_in  = in_f  // n_blocks
        self.block_out = out_f // n_blocks
        self.blocks    = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
        self.mix_down  = nn.Parameter(torch.randn(mix_rank, in_f)  * 0.02)
        self.mix_up    = nn.Parameter(torch.randn(out_f, mix_rank) * 0.02)
    def forward(self, x):
        bsz       = x.shape[:-1]
        x_b       = x.reshape(*bsz, self.n_blocks, self.block_in)
        block_out = torch.einsum('...bi,boi->...bo', x_b, self.blocks)
        block_out = block_out.reshape(*bsz, self.n_blocks * self.block_out)
        mix_out   = (x @ self.mix_down.t()) @ self.mix_up.t()
        return block_out + mix_out
    def param_count(self):
        return self.blocks.numel() + self.mix_down.numel() + self.mix_up.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class GroupedLinear(nn.Module):
    '''Input and output split into g independent groups. Params = in_f * out_f / g.'''
    def __init__(self, in_f, out_f, groups=4):
        super().__init__()
        assert in_f % groups == 0 and out_f % groups == 0
        self.groups    = groups
        self.group_in  = in_f  // groups
        self.group_out = out_f // groups
        self.weight    = nn.Parameter(torch.randn(groups, self.group_out, self.group_in) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        x_g = x.reshape(*bsz, self.groups, self.group_in)
        out = torch.einsum('...gi,goi->...go', x_g, self.weight)
        return out.reshape(*bsz, self.groups * self.group_out)
    def param_count(self):
        return self.weight.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class KroneckerLinear(nn.Module):
    '''W = A x B (Kronecker product). Params = a1*a2 + b1*b2.'''
    def __init__(self, in_f, out_f, a_rows=None, a_cols=None):
        super().__init__()
        if a_rows is None:
            a_rows = int(math.sqrt(out_f))
            while out_f % a_rows != 0:
                a_rows -= 1
        if a_cols is None:
            a_cols = int(math.sqrt(in_f))
            while in_f % a_cols != 0:
                a_cols -= 1
        self.a_rows = a_rows
        self.a_cols = a_cols
        self.b_rows = out_f // a_rows
        self.b_cols = in_f  // a_cols
        self.out_f  = out_f
        self.A      = nn.Parameter(torch.randn(a_rows, a_cols) * 0.02)
        self.B      = nn.Parameter(torch.randn(self.b_rows, self.b_cols) * 0.02)
    def forward(self, x):
        bsz = x.shape[:-1]
        x_r = x.reshape(*bsz, self.a_cols, self.b_cols)
        xB  = x_r @ self.B.t()
        xBA = xB.transpose(-2, -1) @ self.A.t()
        return xBA.transpose(-2, -1).reshape(*bsz, self.out_f)
    def param_count(self):
        return self.A.numel() + self.B.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class HashLinear(nn.Module):
    '''Hash-trick weight sharing: n_buckets unique params, virtual [out, in] via hash.'''
    def __init__(self, in_f, out_f, n_buckets=65536):
        super().__init__()
        self.in_f  = in_f
        self.out_f = out_f
        self.table = nn.Parameter(torch.randn(n_buckets) * 0.02)
        rows = torch.arange(out_f).unsqueeze(1).expand(out_f, in_f)
        cols = torch.arange(in_f).unsqueeze(0).expand(out_f, in_f)
        idx  = ((rows * 1000003 + cols * 999983) ^ (rows * 31 + cols * 7)) % n_buckets
        self.register_buffer('hash_idx', idx.long())
    def forward(self, x):
        return x @ self.table[self.hash_idx].t()
    def param_count(self):
        return self.table.numel()
    def ternary_bytes(self):
        return self.table.numel() * 1.6 / 8


class StructuredSparse(nn.Module):
    '''Full weight matrix with fixed random binary mask. Storage = keep_ratio * full params.'''
    def __init__(self, in_f, out_f, keep_ratio=0.25):
        super().__init__()
        self.weight     = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        self.keep_ratio = keep_ratio
        mask            = torch.zeros(out_f, in_f, dtype=torch.bool)
        n_keep          = int(out_f * in_f * keep_ratio)
        mask.view(-1)[torch.randperm(out_f * in_f)[:n_keep]] = True
        self.register_buffer('mask', mask)
    def forward(self, x):
        return x @ (self.weight * self.mask).t()
    def param_count(self):
        return int(self.mask.sum().item())
    def ternary_bytes(self):
        n_nz       = self.param_count()
        mask_bytes = self.weight.numel() / 8
        return mask_bytes + n_nz * 1.6 / 8


class DualPathLinear(nn.Module):
    '''Dense low-rank path + block-diagonal path. Global + local combined.'''
    def __init__(self, in_f, out_f, dense_rank=32, n_blocks=8):
        super().__init__()
        assert in_f % n_blocks == 0 and out_f % n_blocks == 0
        self.n_blocks  = n_blocks
        self.block_in  = in_f  // n_blocks
        self.block_out = out_f // n_blocks
        self.dense_down = nn.Parameter(torch.randn(dense_rank, in_f)  * 0.02)
        self.dense_up   = nn.Parameter(torch.randn(out_f, dense_rank) * 0.02)
        self.blocks     = nn.Parameter(torch.randn(n_blocks, self.block_out, self.block_in) * 0.02)
    def forward(self, x):
        bsz   = x.shape[:-1]
        dense = (x @ self.dense_down.t()) @ self.dense_up.t()
        x_b   = x.reshape(*bsz, self.n_blocks, self.block_in)
        block = torch.einsum('...bi,boi->...bo', x_b, self.blocks).reshape(*bsz, self.n_blocks * self.block_out)
        return dense + block
    def param_count(self):
        return self.dense_down.numel() + self.dense_up.numel() + self.blocks.numel()
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8


class ButterflyLinear(nn.Module):
    '''
    Butterfly factorization: log2(n) stages of structured sparse matmuls.
    Total params = O(n * log(n)). Requires in_f == out_f and power of 2.
    '''
    def __init__(self, dim):
        super().__init__()
        assert dim > 0 and (dim & (dim - 1)) == 0, 'dim must be a power of 2'
        self.dim      = dim
        self.n_stages = int(math.log2(dim))
        self.factors  = nn.ParameterList([
            nn.Parameter(torch.randn(dim, 2) * 0.02) for _ in range(self.n_stages)
        ])
    def forward(self, x):
        bsz  = x.shape[:-1]
        y    = x.reshape(*bsz, self.dim)
        for stage in range(self.n_stages):
            stride      = 1 << stage
            idx0        = torch.arange(self.dim, device=x.device)
            block_idx   = idx0 // (2 * stride)
            within      = idx0 % (2 * stride)
            is_second   = (within >= stride).long()
            pair_idx    = block_idx * (2 * stride) + within % stride
            f           = self.factors[stage]
            a, b        = f[:, 0], f[:, 1]
            y_new       = y.clone()
            partner     = (pair_idx + stride * (1 - 2 * is_second)).clamp(0, self.dim - 1)
            even        = is_second == 0
            odd         = is_second == 1
            y_new[..., even] = a[even] * y[..., even] + b[even] * y[..., partner[even]]
            y_new[..., odd]  = a[odd]  * y[..., odd]  + b[odd]  * y[..., partner[odd]]
            y = y_new
        return y
    def param_count(self):
        return self.dim * 2 * self.n_stages
    def ternary_bytes(self):
        return self.param_count() * 1.6 / 8

### 6. Mini-Transformer

In [ ]:
# -----------------------------------------------------------------------
# Mini-transformer used in Experiment 11 -- linear layer comparison
# inside a real transformer training loop.
# -----------------------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return F.rms_norm(x, (x.size(-1),)) * self.scale


class Attention(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, make_proj):
        super().__init__()
        self.num_heads    = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim     = dim // num_heads
        self.q_size       = num_heads    * self.head_dim
        self.kv_size      = num_kv_heads * self.head_dim
        self.c_qkv        = nn.Linear(dim, self.q_size + 2 * self.kv_size, bias=False)
        self.proj         = make_proj(dim, dim)

    def forward(self, x):
        B, S, D  = x.shape
        q, k, v  = self.c_qkv(x).split([self.q_size, self.kv_size, self.kv_size], dim=-1)
        q        = q.reshape(B, S, self.num_heads,    self.head_dim).transpose(1, 2)
        k        = k.reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v        = v.reshape(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        rep      = self.num_heads // self.num_kv_heads
        k        = k.repeat_interleave(rep, dim=1)
        v        = v.repeat_interleave(rep, dim=1)
        scale    = 1.0 / math.sqrt(self.head_dim)
        attn     = (q @ k.transpose(-2, -1)) * scale
        mask     = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        attn     = F.softmax(attn.masked_fill(mask, float('-inf')), dim=-1)
        y        = (attn @ v).transpose(1, 2).reshape(B, S, D)
        return self.proj(y)


class MLP(nn.Module):
    def __init__(self, dim, hidden, make_fc, make_proj):
        super().__init__()
        self.fc   = make_fc(dim, hidden)
        self.proj = make_proj(hidden, dim)
    def forward(self, x):
        return self.proj(F.relu(self.fc(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads, hidden,
                 make_attn_proj, make_mlp_fc, make_mlp_proj):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = Attention(dim, num_heads, num_kv_heads, make_attn_proj)
        self.norm2 = RMSNorm(dim)
        self.mlp   = MLP(dim, hidden, make_mlp_fc, make_mlp_proj)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab, dim, n_layers, num_heads, num_kv_heads, hidden,
                 make_attn_proj, make_mlp_fc, make_mlp_proj):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, dim)
        self.blocks  = nn.ModuleList([
            TransformerBlock(dim, num_heads, num_kv_heads, hidden,
                             make_attn_proj, make_mlp_fc, make_mlp_proj)
            for _ in range(n_layers)
        ])
        self.norm    = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab, bias=False)
        self.tok_emb.weight = self.head.weight

    def forward(self, idx):
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

### 7. Training and Evaluation Utilities

In [ ]:
def train_eval(model, train_x, train_y, test_x, test_y,
               n_epochs=100, lr=LR, batch=BATCH, extra_params=None):
    '''
    Train a classification model and return test loss, accuracy, and
    per-class accuracy. extra_params can include shared feature banks that
    are not owned by the model (e.g. shared_feat for TverskySharedCls).
    '''
    model  = model.to(device)
    params = list(model.parameters())
    if extra_params:
        params += [p for p in extra_params if not any(p is q for q in params)]
    opt = torch.optim.Adam(params, lr=lr)

    for _ in range(n_epochs):
        idx = torch.randperm(len(train_x), device=device)
        for i in range(0, len(train_x), batch):
            b = idx[i:i + batch]
            opt.zero_grad()
            F.cross_entropy(model(train_x[b]), train_y[b]).backward()
            opt.step()

    with torch.no_grad():
        logits    = model(test_x)
        loss      = F.cross_entropy(logits, test_y).item()
        preds     = logits.argmax(-1)
        acc       = (preds == test_y).float().mean().item()
        n_classes = int(test_y.max().item()) + 1
        per_class = [(preds[test_y == i] == i).float().mean().item()
                     if (test_y == i).any() else 0.0
                     for i in range(n_classes)]
        avg       = sum(per_class) / n_classes
    return loss, acc, per_class, avg


def benchmark(model, test_x, n_warmup=50, n_runs=500):
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        for _ in range(n_warmup):
            model(test_x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elif device.type == 'mps':
            torch.mps.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(test_x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elif device.type == 'mps':
            torch.mps.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000


def storage_mb(feat_params, proto_params, n_layers=N_LAYERS, shared_feat=True):
    '''fp16 storage in MB for features + prototypes across n_layers.'''
    feat_cost  = feat_params  * 2 * (1 if shared_feat else n_layers)
    proto_cost = proto_params * 2 * n_layers
    return (feat_cost + proto_cost) / 1e6


def storage_linear(out_dim, in_dim, n_layers=N_LAYERS):
    return out_dim * in_dim * 2 * n_layers / 1e6


def storage_ternary(param_count, n_layers=N_LAYERS):
    return param_count * 1.6 / 8 / 1e6 * n_layers


def train_and_eval_transformer(model, train_data, val_data, vocab,
                                n_steps=400, batch=32, lr=3e-4):
    '''Train MiniGPT on next-token prediction and return metrics.'''
    model        = model.to(device)
    opt          = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n_params     = sum(p.numel() for p in model.parameters())
    variant_p    = sum(
        b.attn.proj.param_count() + b.mlp.fc.param_count() + b.mlp.proj.param_count()
        for b in model.blocks
    )
    results      = []

    model.train()
    for _ in range(5):
        idx    = torch.randint(0, len(train_data), (batch,), device=device)
        logits = model(train_data[idx][:, :-1])
        loss   = F.cross_entropy(logits.reshape(-1, vocab), train_data[idx][:, 1:].reshape(-1))
        loss.backward()
        opt.zero_grad()

    if device.type == 'cuda':
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    eval_every = n_steps // 4
    for step in range(1, n_steps + 1):
        model.train()
        idx    = torch.randint(0, len(train_data), (batch,), device=device)
        logits = model(train_data[idx][:, :-1])
        loss   = F.cross_entropy(logits.reshape(-1, vocab), train_data[idx][:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad()
        if step % eval_every == 0 or step == n_steps:
            model.eval()
            with torch.no_grad():
                vl = F.cross_entropy(model(val_data[:, :-1]).reshape(-1, vocab),
                                     val_data[:, 1:].reshape(-1)).item()
            if device.type == 'cuda':
                torch.cuda.synchronize()
            ms = (time.perf_counter() - t_start) / step * 1000
            results.append((step, vl, ms))

    return {
        'val_loss':      results[-1][1],
        'ms_step':       results[-1][2],
        'total_params':  n_params,
        'variant_params': variant_p,
        'ternary_12L_MB': variant_p * 1.6 / 8 / 1e6 * N_LAYERS,
        'history':       results,
    }

### 8. Experiment 1 -- Tversky Feature Count Sweep

In [ ]:
# -----------------------------------------------------------------------
# Config for this experiment.
# -----------------------------------------------------------------------

E1_VOCAB     = 8192
E1_CLUSTERS  = 64
E1_N_TRAIN   = 4000
E1_N_TEST    = 1000
E1_OUT_DIM   = 128

# -----------------------------------------------------------------------
# Build data -- cluster-structured token embeddings.
# -----------------------------------------------------------------------

cluster_centers_e1 = torch.randn(E1_CLUSTERS, DIM) * 2.0
tpc                = E1_VOCAB // E1_CLUSTERS
token_emb_e1       = torch.zeros(E1_VOCAB, DIM)
for c in range(E1_CLUSTERS):
    s, e = c * tpc, min((c + 1) * tpc, E1_VOCAB)
    token_emb_e1[s:e] = cluster_centers_e1[c] + torch.randn(e - s, DIM) * 0.5
token_emb_e1 = F.normalize(token_emb_e1, dim=-1)

freq       = 1.0 / (torch.arange(1, E1_VOCAB + 1).float() ** 0.8)
freq       = freq / freq.sum()
train_ids  = torch.multinomial(freq.expand(E1_N_TRAIN, -1), 2, replacement=True)
test_ids   = torch.multinomial(freq.expand(E1_N_TEST, -1),  2, replacement=True)
train_x_e1 = token_emb_e1[train_ids[:, 0]]
train_y_e1 = train_ids[:, 1] % E1_OUT_DIM
test_x_e1  = token_emb_e1[test_ids[:, 0]]
test_y_e1  = test_ids[:, 1] % E1_OUT_DIM

# -----------------------------------------------------------------------
# Sweep over feature counts.
# -----------------------------------------------------------------------

lin_e1 = LinearProj(DIM, E1_OUT_DIM)
lin_loss_e1, _, _, _ = train_eval(lin_e1, train_x_e1, train_y_e1,
                                   test_x_e1, test_y_e1, n_epochs=50, batch=256)
print(f'Linear : loss={lin_loss_e1:.4f}  params={lin_e1.param_count():,}')
print()

feature_sweep = [4, 8, 16, 32, 64, 128, 256, 512]
sweep_results = []

for nf in feature_sweep:
    tv         = TverskyProj(DIM, E1_OUT_DIM, nf)
    loss, _, _, _ = train_eval(tv, train_x_e1, train_y_e1,
                                test_x_e1, test_y_e1, n_epochs=50, batch=256)
    sweep_results.append({'nf': nf, 'loss': loss, 'params': tv.param_count()})
    delta = loss - lin_loss_e1
    sign  = 'BEATS' if loss < lin_loss_e1 else 'worse'
    print(f'{sign} | {nf:4d} features: loss={loss:.4f} ({delta:+.4f})  params={tv.param_count():,}')

# -----------------------------------------------------------------------
# Speed comparison.
# -----------------------------------------------------------------------

lin_time = benchmark(lin_e1, test_x_e1, n_warmup=10, n_runs=100)
for r in sweep_results:
    tv      = TverskyProj(DIM, E1_OUT_DIM, r['nf'])
    tv_time = benchmark(tv, test_x_e1, n_warmup=10, n_runs=100)
    print(f'{r["nf"]:4d} features: {tv_time:.3f}ms vs linear {lin_time:.3f}ms ({tv_time / lin_time:.1f}x)')

### 9. Experiment 2 -- Directional Classification

In [ ]:
# -----------------------------------------------------------------------
# Config for this experiment.
# -----------------------------------------------------------------------

E2_N_CONCEPTS = 800
E2_N_TRAIN    = 20000
E2_N_TEST     = 4000
E2_N_EPOCHS   = 100
E2_OUT        = 3
E2_IN         = DIM * 2

# -----------------------------------------------------------------------
# Build hierarchical concept data.
# -----------------------------------------------------------------------

concepts_e2 = build_concepts(E2_N_CONCEPTS).to(device)

train_xa2, train_ya2, train_labels2 = make_dir_pairs(concepts_e2, E2_N_TRAIN)
test_xa2,  test_ya2,  test_labels2  = make_dir_pairs(concepts_e2, E2_N_TEST)

train_dir2   = torch.cat([train_xa2, train_ya2], dim=-1).to(device)
test_dir2    = torch.cat([test_xa2,  test_ya2],  dim=-1).to(device)
train_labels2 = train_labels2.to(device)
test_labels2  = test_labels2.to(device)

print(f'Dir task label dist : {[(test_labels2 == i).sum().item() for i in range(E2_OUT)]}')

# -----------------------------------------------------------------------
# Shared feature banks for cross-layer sharing variants.
# -----------------------------------------------------------------------

sf = {nf: nn.Parameter(torch.empty(nf, E2_IN).uniform_(-0.02, 0.02)).to(device)
       for nf in [4, 8, 16, 32, 64]}

models_e2 = {
    'Linear':              (LinearCls(E2_IN, E2_OUT),                            None,   E2_IN * E2_OUT, 0),
    'Tversky sigmoid 4f':  (TverskySigmoid(E2_IN, E2_OUT, 4),                    None,   E2_IN * E2_OUT, 4 * E2_IN),
    'Tversky sigmoid 8f':  (TverskySigmoid(E2_IN, E2_OUT, 8),                    None,   E2_IN * E2_OUT, 8 * E2_IN),
    'Tversky relu 4f':     (TverskyReLU(E2_IN, E2_OUT, 4),                       None,   E2_IN * E2_OUT, 4 * E2_IN),
    'Tversky relu 8f':     (TverskyReLU(E2_IN, E2_OUT, 8),                       None,   E2_IN * E2_OUT, 8 * E2_IN),
    'Tversky noFeatures':  (TverskyNoFeatures(E2_IN, E2_OUT),                    None,   E2_IN * E2_OUT, 0),
    'TverskyNoFeat poly':  (TverskyNoFeatPoly(E2_IN, E2_OUT),                    None,   E2_IN * E2_OUT, 0),
    'Tversky shared 4f':   (TverskySharedCls(E2_IN, E2_OUT, 4,  sf[4]),   [sf[4]],  E2_IN * E2_OUT, 4  * E2_IN),
    'Tversky shared 8f':   (TverskySharedCls(E2_IN, E2_OUT, 8,  sf[8]),   [sf[8]],  E2_IN * E2_OUT, 8  * E2_IN),
    'Tversky shared 16f':  (TverskySharedCls(E2_IN, E2_OUT, 16, sf[16]), [sf[16]], E2_IN * E2_OUT, 16 * E2_IN),
    'Tversky shared 32f':  (TverskySharedCls(E2_IN, E2_OUT, 32, sf[32]), [sf[32]], E2_IN * E2_OUT, 32 * E2_IN),
    'Tversky shared 64f':  (TverskySharedCls(E2_IN, E2_OUT, 64, sf[64]), [sf[64]], E2_IN * E2_OUT, 64 * E2_IN),
}

lin_stor = storage_linear(E2_OUT, E2_IN)
print(f'\n{"Model":<26} {"Loss":>6} {"Acc":>6} {"A>B":>6} {"B>A":>6} {"Unr":>6} {"Avg":>6} {"ms":>6} {"StorMB(12L)":>12}')
print('-' * 96)

for name, (model, extra, proto_p, feat_p) in models_e2.items():
    loss, acc, per_class, avg = train_eval(model, train_dir2, train_labels2,
                                           test_dir2, test_labels2,
                                           n_epochs=E2_N_EPOCHS)
    t    = benchmark(model, test_dir2)
    stor = storage_mb(feat_p, proto_p, shared_feat=(feat_p > 0))
    a0, a1, a2 = per_class
    print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {a0:>6.3f} {a1:>6.3f} {a2:>6.3f} {avg:>6.3f} {t:>5.2f}ms {stor:>10.3f}MB')

print(f'\nLinear storage ({N_LAYERS}L): {lin_stor:.3f}MB')

### 10. Experiment 3 -- Projection Benchmark

In [ ]:
# -----------------------------------------------------------------------
# Config for this experiment.
# -----------------------------------------------------------------------

E3_N_CONCEPTS  = 500
E3_N_TRAIN     = 10000
E3_N_TEST      = 2000
E3_N_EPOCHS    = 80
E3_OUT_CLASSES = 3
E3_DIR_IN      = DIM * 2

# -----------------------------------------------------------------------
# Build data.
# -----------------------------------------------------------------------

concepts_e3 = build_concepts(E3_N_CONCEPTS, feature_noise=0.0).to(device)

train_xa3, train_ya3, train_labels3 = make_dir_pairs(concepts_e3, E3_N_TRAIN)
test_xa3,  test_ya3,  test_labels3  = make_dir_pairs(concepts_e3, E3_N_TEST)
train_dir3   = torch.cat([train_xa3, train_ya3], dim=-1).to(device)
test_dir3    = torch.cat([test_xa3,  test_ya3],  dim=-1).to(device)
train_labels3 = train_labels3.to(device)
test_labels3  = test_labels3.to(device)

freq3 = 1.0 / (torch.arange(1, 8193).float() ** 0.8)
freq3 = freq3 / freq3.sum()
clust_ids3    = torch.arange(8192) % 64
tok_emb3      = torch.randn(8192, DIM)
for c in range(64):
    s, e = c * 128, min((c + 1) * 128, 8192)
    tok_emb3[s:e] = torch.randn(DIM) * 2.0 + torch.randn(e - s, DIM) * 0.5
tok_emb3 = F.normalize(tok_emb3, dim=-1)

train_lm3_x, train_lm3_y = make_lm_pairs(tok_emb3, clust_ids3, freq3, E3_N_TRAIN, out_classes=E3_OUT_CLASSES)
test_lm3_x,  test_lm3_y  = make_lm_pairs(tok_emb3, clust_ids3, freq3, E3_N_TEST,  out_classes=E3_OUT_CLASSES)
train_lm3_x, test_lm3_x  = train_lm3_x.to(device), test_lm3_x.to(device)
train_lm3_y, test_lm3_y  = train_lm3_y.to(device), test_lm3_y.to(device)

proj_test3 = torch.randn(2000, DIM, device=device)

# -----------------------------------------------------------------------
# Projection benchmark -- speed and storage only, no task accuracy.
# -----------------------------------------------------------------------

proj_models3 = {
    'Linear':          StandardLinear(DIM, DIM),
    'LowRank r=64':    LowRankLinear(DIM, DIM, rank=64),
    'LowRank r=128':   LowRankLinear(DIM, DIM, rank=128),
    'LowRank r=256':   LowRankLinear(DIM, DIM, rank=256),
    'LowRank r=384':   LowRankLinear(DIM, DIM, rank=384),
    'BlockDiag b=2':   BlockDiagonalLinear(DIM, DIM, n_blocks=2),
    'BlockDiag b=4':   BlockDiagonalLinear(DIM, DIM, n_blocks=4),
    'BlockDiag b=8':   BlockDiagonalLinear(DIM, DIM, n_blocks=8),
    'BlockDiag b=16':  BlockDiagonalLinear(DIM, DIM, n_blocks=16),
    'BD4 + mix16':     BlockDiagPlusMix(DIM, DIM, n_blocks=4, mix_rank=16),
    'BD4 + mix32':     BlockDiagPlusMix(DIM, DIM, n_blocks=4, mix_rank=32),
    'BD8 + mix16':     BlockDiagPlusMix(DIM, DIM, n_blocks=8, mix_rank=16),
    'BD8 + mix32':     BlockDiagPlusMix(DIM, DIM, n_blocks=8, mix_rank=32),
    'Grouped g=2':     GroupedLinear(DIM, DIM, groups=2),
    'Grouped g=4':     GroupedLinear(DIM, DIM, groups=4),
    'Kronecker':       KroneckerLinear(DIM, DIM),
    'DualPath r32 b8': DualPathLinear(DIM, DIM, dense_rank=32, n_blocks=8),
    'DualPath r64 b4': DualPathLinear(DIM, DIM, dense_rank=64, n_blocks=4),
    'Sparse 25%':      StructuredSparse(DIM, DIM, keep_ratio=0.25),
    'Sparse 50%':      StructuredSparse(DIM, DIM, keep_ratio=0.50),
    'Hash 65536':      HashLinear(DIM, DIM, n_buckets=65536),
    'Hash 131072':     HashLinear(DIM, DIM, n_buckets=131072),
}

print('\n' + '=' * 100)
print('PROJECTION BENCHMARK (DIM -> DIM): Speed + Storage')
print('=' * 100)
print(f'{"Model":<26} {"Params":>10} {"1L MB":>8} {"12L MB":>8} {"ms":>8} {"vs Lin":>8} {"Efficiency":>10}')
print('-' * 90)

lin_spd = lin_par = None
for name, model in proj_models3.items():
    pc  = model.param_count()
    mb1 = pc * 1.6 / 8 / 1e6
    mb12 = mb1 * N_LAYERS
    t   = benchmark(model, proj_test3)
    if lin_spd is None:
        lin_spd, lin_par = t, pc
    speedup     = f'{t / lin_spd:.2f}x'
    efficiency  = f'{mb12 * t:.2f}'
    print(f'{name:<26} {pc:>10,} {mb1:>7.3f}M {mb12:>7.2f}M {t:>7.2f}ms {speedup:>8} {efficiency:>10}')

# -----------------------------------------------------------------------
# Task accuracy -- directional and LM next-token.
# -----------------------------------------------------------------------

dir_models3 = {
    'Linear':        StandardLinear(E3_DIR_IN, E3_OUT_CLASSES),
    'LowRank r=32':  LowRankLinear(E3_DIR_IN, E3_OUT_CLASSES, rank=32),
    'LowRank r=128': LowRankLinear(E3_DIR_IN, E3_OUT_CLASSES, rank=128),
    'Sparse 25%':    StructuredSparse(E3_DIR_IN, E3_OUT_CLASSES, keep_ratio=0.25),
    'Sparse 50%':    StructuredSparse(E3_DIR_IN, E3_OUT_CLASSES, keep_ratio=0.50),
    'Hash 1024':     HashLinear(E3_DIR_IN, E3_OUT_CLASSES, n_buckets=1024),
    'Hash 4096':     HashLinear(E3_DIR_IN, E3_OUT_CLASSES, n_buckets=4096),
}

lm_models3 = {
    'Linear':        StandardLinear(DIM, E3_OUT_CLASSES),
    'LowRank r=32':  LowRankLinear(DIM, E3_OUT_CLASSES, rank=32),
    'LowRank r=128': LowRankLinear(DIM, E3_OUT_CLASSES, rank=128),
    'Sparse 25%':    StructuredSparse(DIM, E3_OUT_CLASSES, keep_ratio=0.25),
    'Sparse 50%':    StructuredSparse(DIM, E3_OUT_CLASSES, keep_ratio=0.50),
    'Hash 512':      HashLinear(DIM, E3_OUT_CLASSES, n_buckets=512),
    'Hash 2048':     HashLinear(DIM, E3_OUT_CLASSES, n_buckets=2048),
}

for task_name, task_models, tx, ty in [
    ('DIRECTIONAL CLASSIFICATION', dir_models3, test_dir3,   test_labels3),
    ('LM NEXT-TOKEN PREDICTION',   lm_models3,  test_lm3_x,  test_lm3_y),
]:
    train_x_ = train_dir3   if 'DIRECTIONAL' in task_name else train_lm3_x
    train_y_ = train_labels3 if 'DIRECTIONAL' in task_name else train_lm3_y
    print('\n' + '=' * 100)
    print(task_name)
    print('=' * 100)
    print(f'{"Model":<26} {"Loss":>6} {"Acc":>6} {"Avg":>6} {"Params":>10} {"12L MB":>8} {"ms":>8}')
    print('-' * 85)
    for name, model in task_models.items():
        loss, acc, _, avg = train_eval(model, train_x_, train_y_, tx, ty,
                                        n_epochs=E3_N_EPOCHS)
        t  = benchmark(model, tx)
        pc = model.param_count()
        mb = storage_ternary(pc)
        print(f'{name:<26} {loss:>6.3f} {acc:>6.3f} {avg:>6.3f} {pc:>10,} {mb:>7.2f}M {t:>7.2f}ms')

### 11. Experiment 4 -- Mini-Transformer Comparison

In [ ]:
# -----------------------------------------------------------------------
# Config for this experiment -- smaller vocab for fast iteration.
# -----------------------------------------------------------------------

E4_VOCAB        = 512
E4_DIM          = DIM
E4_NUM_HEADS    = 8
E4_NUM_KV_HEADS = 4
E4_MLP_MULT     = 2
E4_HIDDEN       = E4_DIM * E4_MLP_MULT
E4_SEQ_LEN      = 128
E4_BATCH        = 32
E4_N_LAYERS     = 3
E4_N_STEPS      = 400
E4_LR           = 3e-4

# -----------------------------------------------------------------------
# Synthetic data for transformer training.
# -----------------------------------------------------------------------

train_data4 = generate_sequences(E4_VOCAB, 16, 2000, E4_SEQ_LEN).to(device)
val_data4   = generate_sequences(E4_VOCAB, 16, 500,  E4_SEQ_LEN).to(device)

# -----------------------------------------------------------------------
# Configurations to test: different layer types for attn_proj and MLP.
# -----------------------------------------------------------------------

configs_e4 = {
    'All Standard':      {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: StandardLinear(i, o),
        'mlp_proj':  lambda i, o: StandardLinear(i, o),
    },
    'MLP Grouped g=2':   {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=2),
    },
    'MLP Grouped g=4':   {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=4),
    },
    'All Grouped g=2':   {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=2),
    },
    'All Grouped g=4':   {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=4),
    },
    'MLP BD4+mix32':     {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: BlockDiagPlusMix(i, o, n_blocks=4, mix_rank=32),
        'mlp_proj':  lambda i, o: BlockDiagPlusMix(i, o, n_blocks=4, mix_rank=32),
    },
    'MLP LowRank r=128': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: LowRankLinear(i, o, rank=128),
        'mlp_proj':  lambda i, o: LowRankLinear(i, o, rank=128),
    },
    'MLP LowRank r=256': {
        'attn_proj': lambda i, o: StandardLinear(i, o),
        'mlp_fc':    lambda i, o: LowRankLinear(i, o, rank=256),
        'mlp_proj':  lambda i, o: LowRankLinear(i, o, rank=256),
    },
    'Attn Grouped g=2':  {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_fc':    lambda i, o: StandardLinear(i, o),
        'mlp_proj':  lambda i, o: StandardLinear(i, o),
    },
    'Attn LowRank r=128': {
        'attn_proj': lambda i, o: LowRankLinear(i, o, rank=128),
        'mlp_fc':    lambda i, o: StandardLinear(i, o),
        'mlp_proj':  lambda i, o: StandardLinear(i, o),
    },
    'Attn g=2, MLP g=4': {
        'attn_proj': lambda i, o: GroupedLinear(i, o, groups=2),
        'mlp_fc':    lambda i, o: GroupedLinear(i, o, groups=4),
        'mlp_proj':  lambda i, o: GroupedLinear(i, o, groups=4),
    },
}

print(f'Mini-Transformer: {E4_N_LAYERS}L {E4_DIM}d {E4_NUM_HEADS}h {E4_NUM_KV_HEADS}kv MLP={E4_MLP_MULT}x')
print(f'Training: {E4_N_STEPS} steps  batch={E4_BATCH}  seq={E4_SEQ_LEN}  vocab={E4_VOCAB}')
print()
print(f'{"Config":<28} {"ValLoss":>8} {"ms/step":>8} {"TotParams":>10} {"VarParams":>10} {"Tern12L":>8} {"Notes":>20}')
print('-' * 105)

all_results4   = []
baseline_loss4 = None

for name, cfg in configs_e4.items():
    model  = MiniGPT(
        E4_VOCAB, E4_DIM, E4_N_LAYERS, E4_NUM_HEADS, E4_NUM_KV_HEADS, E4_HIDDEN,
        make_attn_proj=cfg['attn_proj'],
        make_mlp_fc=cfg['mlp_fc'],
        make_mlp_proj=cfg['mlp_proj'],
    )
    result = train_and_eval_transformer(model, train_data4, val_data4, E4_VOCAB,
                                         n_steps=E4_N_STEPS, batch=E4_BATCH, lr=E4_LR)
    result['label'] = name
    all_results4.append(result)
    if baseline_loss4 is None:
        baseline_loss4 = result['val_loss']
    delta = result['val_loss'] - baseline_loss4
    sign  = '+' if delta >= 0 else ''
    notes = f'{sign}{delta:.4f} vs base'
    print(f'{name:<28} {result["val_loss"]:>8.4f} {result["ms_step"]:>7.1f}ms '
          f'{result["total_params"]:>10,} {result["variant_params"]:>10,} '
          f'{result["ternary_12L_MB"]:>7.2f}M {notes:>20}')

print('\n' + '=' * 105)
print('TRAINING CURVES (val_loss at checkpoints)')
print('=' * 105)
for r in all_results4:
    curve = ' | '.join([f'step {s}: {l:.4f}' for s, l, _ in r['history']])
    print(f'{r["label"]:<28} {curve}')